# 🧪 Notebook 05 — Evaluate & Deploy

**Goal**: Run **batch evaluation** on agent responses, review results, and understand the **hosted agent** deployment path using `azure-ai-agentserver-agentframework`.

**Capabilities demonstrated**:
- Batch evaluation with `azure-ai-evaluation` SDK
- Built-in evaluators: fluency, coherence, groundedness, violence
- Hosted agent packaging with `from_agent_framework()`
- Deployment via `azd` or VS Code extension

## Prerequisites
- `.env` with `AZURE_AI_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Optionally `AZURE_AI_PROJECT` for Foundry portal logging
- `pip install azure-ai-evaluation azure-ai-agentserver-agentframework`

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os, json
from pathlib import Path
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from utils.config import get_foundry_client, get_env
from agent_framework import Agent

client = get_foundry_client()
print("✅ Connected to Foundry")

---
## Step 1 — Create a test agent & generate responses

In [ ]:
# ── 1a. Create agent ─────────────────────────────────────────────────────────
agent = Agent(
    client=client,
    name="EvalAgent",
    instructions=(
        "You are a helpful assistant that answers general knowledge questions. "
        "Be accurate, concise, and well-structured."
    ),
)
print(f"🤖 Agent: {agent.name}")

In [ ]:
# ── 1b. Test dataset ─────────────────────────────────────────────────────────
test_queries = [
    {
        "query": "What is the capital of France?",
        "ground_truth": "The capital of France is Paris.",
    },
    {
        "query": "Explain photosynthesis in two sentences.",
        "ground_truth": (
            "Photosynthesis is the process by which plants convert sunlight, "
            "water, and carbon dioxide into glucose and oxygen. It occurs "
            "primarily in the chloroplasts of plant cells."
        ),
    },
    {
        "query": "What are the three laws of thermodynamics?",
        "ground_truth": (
            "1) Energy cannot be created or destroyed. "
            "2) Entropy of an isolated system always increases. "
            "3) Entropy approaches a constant as temperature approaches absolute zero."
        ),
    },
    {
        "query": "Who wrote Romeo and Juliet?",
        "ground_truth": "William Shakespeare wrote Romeo and Juliet.",
    },
]
print(f"📋 Test dataset: {len(test_queries)} queries")

In [ ]:
# ── 1c. Generate agent responses ─────────────────────────────────────────────
eval_data = []

for item in test_queries:
    result = await agent.run(item["query"])
    eval_data.append({
        "query": item["query"],
        "response": str(result),
        "ground_truth": item["ground_truth"],
    })
    print(f"✅ {item['query'][:50]}...")

# Save to JSONL for evaluation
eval_file = Path("assets/eval_data.jsonl")
with eval_file.open("w") as f:
    for row in eval_data:
        f.write(json.dumps(row) + "\n")

print(f"\n💾 Saved {len(eval_data)} rows to {eval_file}")

---
## Step 2 — Run batch evaluation

In [ ]:
# ── 2a. Configure evaluators ─────────────────────────────────────────────────
from azure.ai.evaluation import (
    evaluate,
    FluencyEvaluator,
    CoherenceEvaluator,
    GroundednessEvaluator,
    ViolenceEvaluator,
)

MODEL = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")
AZURE_AI_PROJECT_URL = get_env("AZURE_AI_PROJECT", required=False)

evaluators = {
    "fluency": FluencyEvaluator(model_config={"azure_deployment": MODEL}),
    "coherence": CoherenceEvaluator(model_config={"azure_deployment": MODEL}),
    "groundedness": GroundednessEvaluator(model_config={"azure_deployment": MODEL}),
    "violence": ViolenceEvaluator(model_config={"azure_deployment": MODEL}),
}
print(f"📊 Evaluators: {list(evaluators.keys())}")

In [ ]:
# ── 2b. Run evaluation ───────────────────────────────────────────────────────
eval_kwargs = dict(
    data=str(eval_file),
    evaluation_name="agent-framework-demo-eval",
    evaluators=evaluators,
)
if AZURE_AI_PROJECT_URL and not AZURE_AI_PROJECT_URL.startswith("<"):
    eval_kwargs["azure_ai_project"] = AZURE_AI_PROJECT_URL

print("⏳ Running batch evaluation...")
result = evaluate(**eval_kwargs)
print("✅ Evaluation complete!")

In [ ]:
# ── 2c. Display results ──────────────────────────────────────────────────────
print("\n📊 AGGREGATE METRICS")
print("=" * 50)
for metric, score in result.get("metrics", {}).items():
    print(f"  {metric:30s} {score}")

if result.get("studio_url"):
    print(f"\n🔗 View in Foundry portal: {result['studio_url']}")

print("\n📋 PER-QUERY RESULTS")
print("=" * 50)
for i, row in enumerate(result.get("rows", [])):
    print(f"\n  Query {i+1}: {row.get('inputs.query', 'N/A')[:60]}")
    for k, v in row.items():
        if "outputs." in k:
            print(f"    {k.replace('outputs.', ''):25s} {v}")

---
## Step 3 — Hosted Agent Deployment

The `azure-ai-agentserver-agentframework` package lets you wrap **any** Agent Framework agent into a container that Foundry Agent Service can host.

### How it works

```python
# app.py — your hosted agent entry point
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.ai.agentserver.agentframework import from_agent_framework
from azure.identity import DefaultAzureCredential
import os

# Build your agent (same code as notebooks!)
client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=DefaultAzureCredential(),
)

agent = Agent(
    client=client,
    name="ContosoConcierge",
    instructions="You are a helpful Contoso assistant.",
    tools=[...],  # all your tools work here
)

# This single line hosts the agent on http://localhost:8088
from_agent_framework(agent).run()
```

### Deploy with Azure Developer CLI

```bash
# 1. Install azd (if needed)
# https://learn.microsoft.com/azure/developer/azure-developer-cli/install-azd

# 2. Initialize from a Foundry hosted agent sample
azd ai agent init

# 3. Provision Azure resources (ACR, Foundry, etc.)
azd provision

# 4. Build container and deploy
azd deploy
```

### Deploy with VS Code

1. `Ctrl+Shift+P` → **Microsoft Foundry: Create new Hosted Agent**
2. Choose Single Agent or Multi-Agent Workflow template
3. Select your model → deploy

Your agent will be available in the **Foundry Playground** and via REST API.

📚 [Quickstart: Deploy your first hosted agent](https://learn.microsoft.com/azure/foundry/agents/quickstarts/quickstart-hosted-agent)

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
try:
    eval_file.unlink()
    print("🧹 Deleted eval_data.jsonl")
except Exception:
    pass
print("✅ All done!")

---
## ✅ What you demonstrated
| Capability | Feature |
|---|---|
| Batch evaluation | `azure-ai-evaluation` SDK with quality + safety evaluators |
| Foundry portal logging | Results visualized in Foundry UI |
| Hosted agent packaging | `from_agent_framework(agent).run()` |
| Container deployment | `azd provision && azd deploy` |

---
## 🎉 Demo Complete!

| Notebook | What you built | Agent Framework Features |
|---|---|---|
| **01** | Basic agent with tracing | `AIProjectClient` → `FoundryAgent`, sessions, streaming |
| **02** | Tool-augmented agent | Code Interpreter, File Search, Web Search, `@tool` |
| **03** | Enterprise integration | Hosted MCP, `FoundryAgent`, `agent.as_tool()` |
| **04** | Multi-agent orchestration | `SequentialBuilder`, `WorkflowBuilder`, sub-workflows |
| **05** | Evaluation & deployment | Batch eval, `from_agent_framework()`, hosted deploy |